# Phase 2 - Step 7: Model Comparison

This notebook benchmarks Logistic Regression, Random Forest, and XGBoost on identical train/test splits, tracking Precision, Recall, F1, ROC-AUC, and training times.

In [1]:
import pandas as pd
import numpy as np
import time
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

processed_dir = Path("data/processed")
df = pd.read_csv(processed_dir / "employee_attrition_processed.csv")

# Feature engineering
df['income_per_year_at_company'] = df['MonthlySalary'] * 12.0 / (df['YearsAtCompany'] + 1.0)
df['promotion_gap_ratio'] = (2026.0 - df['LastPromotionYear']) / (df['YearsAtCompany'] + 1.0)
df['overtime_ratio'] = df['OvertimeHoursPerMonth'] / 160.0
df['leave_utilization'] = df['LeavesTaken'] / 20.0
df['work_life_satisfaction'] = df['WorkLifeBalanceScore'] * df['CustomerSatisfaction']

target_col = 'AttritionRisk'
y = df[target_col].map({'Yes': 1, 'No': 0}).values
drop_cols = ['EmployeeID', 'Name', 'PhoneNumber', 'JoiningDate', 'LastLeaveDate', target_col, 'CountryCode']
X = df.drop(columns=[c for c in drop_cols if c in df.columns])

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
    ]
)

pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

candidate_models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "XGBoost": XGBClassifier(scale_pos_weight=pos_weight, eval_metric='logloss', random_state=42)
}

comparison_results = []
trained_pipelines = {}

for name, clf in candidate_models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    
    start_time = time.time()
    pipe.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    trained_pipelines[name] = pipe
    comparison_results.append({
        "Model": name,
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "ROC-AUC": round(auc, 4),
        "Training Time (s)": round(train_time, 4)
    })

comparison_df = pd.DataFrame(comparison_results)
print("=== MODEL COMPARISON TABLE ===")
print(comparison_df.to_string(index=False))


=== MODEL COMPARISON TABLE ===
              Model  Precision  Recall     F1  ROC-AUC  Training Time (s)
Logistic Regression        0.8  0.7273 0.7619   0.9837             0.0586
      Random Forest        1.0  0.9091 0.9524   1.0000             0.2038
            XGBoost        1.0  1.0000 1.0000   1.0000             0.2587


In [2]:
# Model Selection Rationale
# Rank primarily on Recall, F1, and ROC-AUC
best_model_row = comparison_df.sort_values(by=['Recall', 'F1', 'ROC-AUC'], ascending=False).iloc[0]
best_model_name = best_model_row['Model']

print(f"\nSELECTED BEST MODEL: {best_model_name}")
print(f"Rationale: {best_model_name} achieves superior Recall ({best_model_row['Recall']}) and F1-Score ({best_model_row['F1']}) with robust ROC-AUC ({best_model_row['ROC-AUC']}), minimizing false negatives in identifying at-risk talent.")



SELECTED BEST MODEL: XGBoost
Rationale: XGBoost achieves superior Recall (1.0) and F1-Score (1.0) with robust ROC-AUC (1.0), minimizing false negatives in identifying at-risk talent.
